# Reference mappings
Review source masters and candidate crosswalks for customers, products, warehouses, and companies. Validate the source ID, source context, and canonical ID before approving a relationship. Run 01 first and clear its integrity exceptions before using suspect data.

In [1]:
import pandas as pd
from IPython.display import display

from bronze_review import decode_file, inventory, mapping_candidates, open_bronze

bronze = open_bronze()
catalog = inventory(bronze)

latest = {}
manifests = sorted(
    (row for row in catalog if row["kind"] == "batch manifest"),
    key=lambda row: (row["published_at"], row["run_id"]),
)
for manifest in manifests:
    latest[manifest["source"]] = manifest["run_id"]

def review(domain, master_collections):
    masters = [
        row for row in catalog if row["kind"] == "snapshot"
        and row["run_id"] == latest[row["source"]]
        and row["collection"] in master_collections
    ]
    candidates = mapping_candidates(catalog, domain)
    if not masters:
        print("No matching master collections found.")
    if not candidates:
        print("No candidate crosswalk files found.")
    tables = {}
    for row in masters + candidates:
        print(f'\n{row["kind"]} | {row["source"]} | {row["collection"]} | {row.get("status")}')
        print(row["key"])
        if row.get("status") != "ok":
            continue
        records, fields = decode_file(row["key"], bronze.read(row["key"]))
        if records is None:
            print("Open the raw file in its native reader.")
            continue
        table = pd.json_normalize(records, max_level=1) if records else pd.DataFrame(columns=fields)
        if row["kind"] == "inbound file" and "domain" in table.columns:
            table = table.loc[table["domain"].eq(domain)]
            if table.empty:
                print(f"No {domain} rows in this candidate file.")
                continue
        tables[row["key"]] = table
        display(table.head(20))
    return tables

## Customer
Check source customer key, entity scope, and canonical customer ID.

In [2]:
customer_tables = review("customer", {"Customer", "customers"})


snapshot | dealer-sql | customers | ok
batch/dealer-sql/manual__2026-11-21T15%3A28%3A03.410708%2B00%3A00/customers-efdd47916690be6a971fe6c4cfb3b64b12bf67519c2d27c113551bf3ebb3217e.jsonl


,id,name,registration_key
0,D-C1,Metro Ind. Services,DEMO-100
1,D-C2,Pioneer Engineering,DEMO-200
2,D-C3,Apex Industrial,None
3,D-C4,Summit Process,None
4,D-C5,Valley Fabrication,DEMO-600



snapshot | erpnext | Customer | ok
batch/erpnext/manual__2026-11-21T15%3A27%3A44.914717%2B00%3A00/Customer-2f0d92941e37ba4f8285b7fedea1e40642d1281e2147fa1beef483590edb6fdf.jsonl


,accounts,allowed_companies,companies,creation,credit_limits,customer_group,customer_name,customer_type,default_commission_rate,disabled,...,naming_series,on_hold,owner,portal_users,restrict_to_companies,sales_team,so_required,supplier_numbers,tax_id,territory
0,[],[],[],2026-11-21 16:56:37.225877,[],Commercial,OTC P-C1 Metro Industrial Services,Company,0.0,0,...,CUST-.YYYY.-,0,kirannc78@gmail.com,[],0,[],0,[],DEMO-100,India
1,[],[],[],2026-11-21 16:56:37.284607,[],Commercial,OTC P-C2 Pioneer Engineering,Company,0.0,0,...,CUST-.YYYY.-,0,kirannc78@gmail.com,[],0,[],0,[],DEMO-200,India
2,[],[],[],2026-11-21 16:56:37.332886,[],Commercial,OTC P-C3 Apex Industrial Supply,Company,0.0,0,...,CUST-.YYYY.-,0,kirannc78@gmail.com,[],0,[],0,[],DEMO-300,India
3,[],[],[],2026-11-21 16:56:37.368415,[],Commercial,OTC P-C4 Summit Process Systems,Company,0.0,0,...,CUST-.YYYY.-,0,kirannc78@gmail.com,[],0,[],0,[],DEMO-400,India
4,[],[],[],2026-11-21 16:56:37.404861,[],Commercial,OTC P-C5 Harbor Utilities,Company,0.0,0,...,CUST-.YYYY.-,0,kirannc78@gmail.com,[],0,[],0,[],DEMO-500,India



inbound file | dealer/exports/dealer-export-20261121165650493 | dealer/exports/dealer-export-20261121165650493/storefront_orders-b29de1d5131ed05b69e922dc75a5955fb2500344186498100675205e754d0ce2.csv | ok
files/otc-inbound/dealer/exports/dealer-export-20261121165650493/9dc745b7a550acffd31fb32c21a4a89b724ce259c274a047a6de03e9e90f4a47/objects/dealer/exports/dealer-export-20261121165650493/storefront_orders-b29de1d5131ed05b69e922dc75a5955fb2500344186498100675205e754d0ce2.csv


,id,customer_id,accepted_at
0,D-S1,D-C1,2026-11-05 03:30:00+00:00


... (6 outputs left out)


inbound file | seed-v1 | seed-v1/mdm/candidates.json | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/candidates.json


,domain,parent,dealer,status,expected_group_id,expected_dealer_group_id
0,customer,P-C1,D-C1,approved,G-CUST-001,G-CUST-001
1,customer,P-C2,D-C2,approved,G-CUST-002,G-CUST-002
2,customer,P-C3,D-C3,pending,G-CUST-003,G-CUST-003
3,customer,P-C4,D-C4,pending,G-CUST-004,G-CUST-004
4,customer,P-C5,D-C5,separate,G-CUST-005,G-CUST-006



inbound file | seed-v1 | seed-v1/mdm/evidence/customer-registration.txt | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/evidence/customer-registration.txt
Open the raw file in its native reader.

inbound file | seed-v1 | seed-v1/mdm/account-mappings.json | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/account-mappings.json


,parent.Sales - OPD,parent.Debtors - OPD,parent.45685245 - OTC Bank - OPD,parent.Stock In Hand - OPD,parent.Cost of Goods Sold - OPD,dealer.400-D,dealer.110-D,dealer.100-D,dealer.120-D,dealer.500-D
0,SALES_REVENUE,TRADE_AR,CASH_BANK,INVENTORY,COGS,SALES_REVENUE,TRADE_AR,CASH_BANK,INVENTORY,COGS


## Product
Check source item code or SKU and canonical product ID. Use UOM as supporting context.

In [3]:
product_tables = review("product", {"Item", "products", "UOM"})


snapshot | dealer-sql | products | ok
batch/dealer-sql/manual__2026-11-21T15%3A28%3A03.410708%2B00%3A00/products-3d26b5092c245dd301505ac090d9a11474513c691952de5ebc260faeb546774a.jsonl


,id,manufacturer,name,part,uom
0,D-FLT,None,Filter,F200,Each
1,D-HOSE,ACME,Hydraulic hose,H900,Each
2,D-PUMP,ACME,Replacement pump,RP100,Each
3,D-SEAL,ACME,Seal,S20,Pack
4,D-VALVE,ACME,Valve,V50,Each



snapshot | erpnext | Item | ok
batch/erpnext/manual__2026-11-21T15%3A27%3A44.914717%2B00%3A00/Item-3d09cbe78630512145d280735c855969f2d8ef73ab5ec6332303621a22e081f2.jsonl


,allow_alternative_item,allow_negative_stock,allowed_companies,attributes,auto_create_assets,barcodes,create_new_batch,creation,customer_code,customer_items,...,stock_uom,supplier_items,taxes,total_projected_qty,uoms,use_serial_no_wise_valuation,valuation_method,valuation_rate,variant_based_on,weight_per_unit
0,0,0,[],[],0,[],0,2026-11-21 16:56:37.738709,,[],...,Nos,[],[],0.0,"[{'conversion_factor': 1.0, 'creation': '2026-...",1,,0.0,Item Attribute,0.0
1,0,0,[],[],0,[],0,2026-11-21 16:56:37.784598,,[],...,Nos,[],[],0.0,"[{'conversion_factor': 1.0, 'creation': '2026-...",1,,0.0,Item Attribute,0.0
2,0,0,[],[],0,[],0,2026-11-21 16:56:37.509476,,[],...,Nos,[],[],0.0,"[{'conversion_factor': 1.0, 'creation': '2026-...",1,,0.0,Item Attribute,0.0
3,0,0,[],[],0,[],0,2026-11-21 16:56:37.762011,,[],...,Nos,[],[],0.0,"[{'conversion_factor': 1.0, 'creation': '2026-...",1,,0.0,Item Attribute,0.0
4,0,0,[],[],0,[],0,2026-11-21 16:56:37.710813,,[],...,Nos,[],[],0.0,"[{'conversion_factor': 1.0, 'creation': '2026-...",1,,0.0,Item Attribute,0.0



snapshot | erpnext | UOM | ok
batch/erpnext/manual__2026-11-21T15%3A27%3A44.914717%2B00%3A00/UOM-07e51de5ebe9af78e08890b8be6a958feb8076f1e1daffdd9ccc94b4e56f3d03.jsonl


,category,creation,docstatus,doctype,enabled,idx,modified,modified_by,must_be_whole_number,name,owner,uom_name,common_code,symbol
0,Electric Current,2026-11-21 16:23:40.742828,0,UOM,1,0,2026-11-21 16:23:40.742828,Administrator,0,Abampere,Administrator,Abampere,NaN,NaN
1,Area,2026-11-21 16:23:40.749109,0,UOM,1,0,2026-11-21 16:23:40.749109,Administrator,0,Acre,Administrator,Acre,NaN,NaN
2,Area,2026-11-21 16:23:40.751032,0,UOM,1,0,2026-11-21 16:23:40.751032,Administrator,0,Acre (US),Administrator,Acre (US),NaN,NaN
3,Electric Current,2026-11-21 16:23:40.752493,0,UOM,1,0,2026-11-21 16:23:40.752493,Administrator,0,Ampere,Administrator,Ampere,NaN,NaN
4,Electrical Charge,2026-11-21 16:23:40.754156,0,UOM,1,0,2026-11-21 16:23:40.754156,Administrator,0,Ampere-Hour,Administrator,Ampere-Hour,NaN,NaN
5,Electrical Charge,2026-11-21 16:23:40.755797,0,UOM,1,0,2026-11-21 16:23:40.755797,Administrator,0,Ampere-Minute,Administrator,Ampere-Minute,NaN,NaN
6,Electrical Charge,2026-11-21 16:23:40.757334,0,UOM,1,0,2026-11-21 16:23:40.757334,Administrator,0,Ampere-Second,Administrator,Ampere-Second,NaN,NaN
7,Area,2026-11-21 16:23:40.758727,0,UOM,1,0,2026-11-21 16:23:40.758727,Administrator,0,Are,Administrator,Are,NaN,NaN
8,NaN,2026-11-21 16:23:40.760122,0,UOM,1,0,2026-11-21 16:23:40.760122,Administrator,0,Area,Administrator,Area,NaN,NaN
9,Length,2026-11-21 16:23:40.761817,0,UOM,1,0,2026-11-21 16:23:40.761817,Administrator,0,Arshin,Administrator,Arshin,NaN,NaN


... (8 outputs left out)


inbound file | seed-v1 | seed-v1/mdm/candidates.json | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/candidates.json


,domain,parent,dealer,status,expected_group_id,expected_dealer_group_id
5,product,P-PUMP,D-PUMP,approved,G-PROD-001,G-PROD-001
6,product,P-VALVE,D-VALVE,approved,G-PROD-002,G-PROD-002
7,product,P-FILT,D-FLT,pending,G-PROD-003,G-PROD-003
8,product,P-SEAL,D-SEAL,pending,G-PROD-004,G-PROD-004
9,product,P-GAUGE,D-HOSE,separate,G-PROD-005,G-PROD-006



inbound file | seed-v1 | seed-v1/mdm/account-mappings.json | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/account-mappings.json


,parent.Sales - OPD,parent.Debtors - OPD,parent.45685245 - OTC Bank - OPD,parent.Stock In Hand - OPD,parent.Cost of Goods Sold - OPD,dealer.400-D,dealer.110-D,dealer.100-D,dealer.120-D,dealer.500-D
0,SALES_REVENUE,TRADE_AR,CASH_BANK,INVENTORY,COGS,SALES_REVENUE,TRADE_AR,CASH_BANK,INVENTORY,COGS


## Warehouse
Check source location ID, company scope, and canonical warehouse ID.

In [4]:
warehouse_tables = review("warehouse", {"Warehouse", "warehouses"})


snapshot | dealer-sql | warehouses | ok
batch/dealer-sql/manual__2026-11-21T15%3A28%3A03.410708%2B00%3A00/warehouses-9768ff6ff11fac38f35a3343a4cf4876341f3bc6688a62285a5575befbbcee8b.jsonl


,id,name
0,D-WH1,Dealer distribution



snapshot | erpnext | Warehouse | ok
batch/erpnext/manual__2026-11-21T15%3A27%3A44.914717%2B00%3A00/Warehouse-d8b38dea7383f391187c7bed9cffeec43f17138c40906e52757b18198449b83a.jsonl


,company,creation,disabled,docstatus,doctype,idx,is_group,is_rejected_warehouse,lft,modified,modified_by,name,old_parent,owner,rgt,warehouse_name,parent_warehouse,warehouse_type
0,OTC Parent Demo,2026-11-21 16:26:58.704277,0,0,Warehouse,1,1,0,11,2026-11-21 16:26:58.704277,kirannc78@gmail.com,All Warehouses - OPD,,kirannc78@gmail.com,24,All Warehouses,NaN,NaN
1,OTC Parent Demo,2026-11-21 16:26:58.749432,0,0,Warehouse,0,0,0,16,2026-11-21 16:26:58.749432,kirannc78@gmail.com,Finished Goods - OPD,All Warehouses - OPD,kirannc78@gmail.com,17,Finished Goods,All Warehouses - OPD,NaN
2,OTC Parent Demo,2026-11-21 16:26:58.758998,0,0,Warehouse,0,0,0,18,2026-11-21 16:26:58.758998,kirannc78@gmail.com,Goods In Transit - OPD,All Warehouses - OPD,kirannc78@gmail.com,19,Goods In Transit,All Warehouses - OPD,Transit
3,OTC Parent Demo,2026-11-21 16:56:37.050412,0,0,Warehouse,1,0,0,20,2026-11-21 16:56:37.050412,kirannc78@gmail.com,P-WH1 - OPD,All Warehouses - OPD,kirannc78@gmail.com,21,P-WH1,All Warehouses - OPD,NaN
4,OTC Parent Demo,2026-11-21 16:56:37.114985,0,0,Warehouse,0,0,0,22,2026-11-21 16:56:37.114985,kirannc78@gmail.com,P-WH2 - OPD,All Warehouses - OPD,kirannc78@gmail.com,23,P-WH2,All Warehouses - OPD,NaN
5,OTC Parent Demo,2026-11-21 16:26:58.733705,0,0,Warehouse,0,0,0,12,2026-11-21 16:26:58.733705,kirannc78@gmail.com,Stores - OPD,All Warehouses - OPD,kirannc78@gmail.com,13,Stores,All Warehouses - OPD,NaN
6,OTC Parent Demo,2026-11-21 16:26:58.741968,0,0,Warehouse,0,0,0,14,2026-11-21 16:26:58.741968,kirannc78@gmail.com,Work In Progress - OPD,All Warehouses - OPD,kirannc78@gmail.com,15,Work In Progress,All Warehouses - OPD,NaN



inbound file | seed-v1 | seed-v1/mdm/candidates.json | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/candidates.json
No warehouse rows in this candidate file.

inbound file | seed-v1 | seed-v1/mdm/account-mappings.json | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/account-mappings.json


,parent.Sales - OPD,parent.Debtors - OPD,parent.45685245 - OTC Bank - OPD,parent.Stock In Hand - OPD,parent.Cost of Goods Sold - OPD,dealer.400-D,dealer.110-D,dealer.100-D,dealer.120-D,dealer.500-D
0,SALES_REVENUE,TRADE_AR,CASH_BANK,INVENTORY,COGS,SALES_REVENUE,TRADE_AR,CASH_BANK,INVENTORY,COGS


## Company
Check the ERPNext entity and any inbound entity aliases or IDs. Dealer SQL does not capture a company table.

In [5]:
company_tables = review("company", {"Company"})


snapshot | erpnext | Company | ok
batch/erpnext/manual__2026-11-21T15%3A27%3A44.914717%2B00%3A00/Company-2cfdfec0a9c02f796ebb65e810c1e1b8b3893c36fb4ff1f724bfad80d6859dd4.jsonl


,abbr,accumulated_depreciation_account,allow_account_creation_against_child_company,asset_received_but_not_billed,auto_err_frequency,auto_exchange_rate_revaluation,book_advance_payments_in_separate_party_account,capital_work_in_progress_account,chart_of_accounts,company_name,...,reporting_currency,rgt,round_off_account,round_off_cost_center,stock_adjustment_account,stock_received_but_not_billed,submit_err_jv,total_monthly_sales,valuation_method,write_off_account
0,OPD,Accumulated Depreciation - OPD,0,Asset Received But Not Billed - OPD,Daily,0,0,CWIP Account - OPD,Standard,OTC Parent Demo,...,INR,4,Round Off - OPD,Main - OPD,Stock Adjustment - OPD,Stock Received But Not Billed - OPD,0,0.0,FIFO,Write Off - OPD



inbound file | seed-v1 | seed-v1/mdm/candidates.json | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/candidates.json
No company rows in this candidate file.

inbound file | seed-v1 | seed-v1/mdm/account-mappings.json | ok
files/otc-inbound/seed-v1/8fab9f89f5242c6819f2ea008f6ddba83b9de4370232084fba58917dec72de4e/objects/seed-v1/mdm/account-mappings.json


,parent.Sales - OPD,parent.Debtors - OPD,parent.45685245 - OTC Bank - OPD,parent.Stock In Hand - OPD,parent.Cost of Goods Sold - OPD,dealer.400-D,dealer.110-D,dealer.100-D,dealer.120-D,dealer.500-D
0,SALES_REVENUE,TRADE_AR,CASH_BANK,INVENTORY,COGS,SALES_REVENUE,TRADE_AR,CASH_BANK,INVENTORY,COGS


Record the selected source object, source key columns, and canonical key column for each approved mapping. Check for blank keys, duplicate scoped keys, conflicting targets, and unmapped source records when the crosswalk is confirmed.